In [1]:
import torch
import torch.nn as nn

In [2]:
import numpy as np
import os
import json

# 데이터 경로 설정
data_dir = '/Users/soyun/Desktop/au_deepfake/AU_Deepfake/data/processed'

# 예시로 첫 번째 폴더의 데이터 로드
sample_folder = os.path.join(data_dir, 'vid_000000_26__walking_down_street_outside_angry')
au_sequence_path = os.path.join(sample_folder, 'au_sequence.npy')
meta_path = os.path.join(sample_folder, 'meta.json')

# AU 시퀀스 로드
au_sequence = np.load(au_sequence_path)
print("AU sequence shape:", au_sequence.shape)

# 메타데이터 로드
with open(meta_path, 'r') as f:
    meta = json.load(f)
print("Meta data:", meta)

AU sequence shape: (64, 17)
Meta data: {'video_id': 'vid_000000_26__walking_down_street_outside_angry', 'label': 0, 'source': 'DFD', 'n_original_frames': 64, 'target_frames': 64, 'au_shape': [64, 17], 'video_path': '/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences/26__walking_down_street_outside_angry.mp4'}


In [3]:
class AUDeepfakeDetector(nn.Module):
    def __init__(self, input_dim=17, seq_len=64, hidden_dim=64, num_classes=1):
        super(AUDeepfakeDetector, self).__init__()

        # 1. 1D CNN: 지역적인 AU 패턴 변화(미세한 떨림 등) 추출
        # 입력 형태: (Batch, Channels, Length)를 맞추기 위해 forward에서 transpose 필요
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2) # Length: 64 -> 32
        
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        # 2nd pool 적용 시 Length: 32 -> 16

        # 2. Bi-LSTM: 표정 변화의 시간적 비일관성 탐지
        self.lstm = nn.LSTM(
            input_size=64, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            bidirectional=True,
            dropout=0.3
        )

        # 3. Classifier
        # Bi-LSTM이므로 출력 차원은 hidden_dim * 2
        self.fc1 = nn.Linear(hidden_dim * 2, 32)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(32, num_classes)
        
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape from dataset: (B, 64, 17)
        # Conv1d는 (B, C, L) 포맷을 요구하므로 transpose
        x = x.transpose(1, 2)  # -> (B, 17, 64)

        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)       # -> (B, 32, 32)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)       # -> (B, 64, 16)

        # LSTM 입력을 위해 (B, L, C) 포맷으로 복구
        x = x.transpose(1, 2)  # -> (B, 16, 64)

        # lstm_out shape: (B, L, hidden_dim * 2)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # 시퀀스의 마지막 타임스텝 출력값만 분류기에 전달
        last_hidden = lstm_out[:, -1, :] # -> (B, hidden_dim * 2)

        out = self.fc1(last_hidden)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        # 학습 시 BCEWithLogitsLoss를 사용한다면 sigmoid를 제거하는 것이 수치적으로 안정적임.
        # 추론 시 확률값이 필요하므로 일단 적용해 둠.
        return self.sigmoid(out)

In [8]:
class LightweightAUDetector(nn.Module):
    def __init__(self, input_dim=17, seq_len=64, hidden_dim=64, num_classes=1):
        super(LightweightAUDetector, self).__init__()

        # 간단한 1D CNN만 사용
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)  # Global average pooling

        self.fc = nn.Linear(32, num_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, 17, 64)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)  # (B, 32, 1)
        x = x.squeeze(-1)  # (B, 32)
        x = self.fc(x)
        return self.sigmoid(x)

In [4]:
from torch.utils.data import Dataset, DataLoader
import torch

class AUDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.samples = []
        
        # 모든 폴더를 순회하며 샘플 수집
        for folder in os.listdir(data_dir):
            if folder.startswith('vid_'):
                folder_path = os.path.join(data_dir, folder)
                au_path = os.path.join(folder_path, 'au_sequence.npy')
                meta_path = os.path.join(folder_path, 'meta.json')
                
                if os.path.exists(au_path) and os.path.exists(meta_path):
                    self.samples.append((au_path, meta_path))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        au_path, meta_path = self.samples[idx]
        
        # AU 시퀀스 로드
        au_sequence = np.load(au_path).astype(np.float32)  # (64, 17)
        
        # 메타데이터 로드
        with open(meta_path, 'r') as f:
            meta = json.load(f)
        
        label = meta['label']
        
        return au_sequence, label

# 데이터셋 생성
dataset = AUDataset(data_dir)
print("Total samples:", len(dataset))

# DataLoader 생성
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# 샘플 배치 확인
for batch in dataloader:
    au_batch, labels = batch
    print("Batch AU shape:", au_batch.shape)  # (8, 64, 17)
    print("Batch labels:", labels)
    break

Total samples: 199
Batch AU shape: torch.Size([8, 64, 17])
Batch labels: tensor([0, 0, 0, 0, 0, 0, 0, 0])


In [5]:
# 텐서 형태 확인용 더미 테스트
if __name__ == "__main__":
    batch_size = 8
    # 전처리 완료된 (B, T, C) 형태의 AU 텐서
    mock_au_input = torch.rand(batch_size, 64, 17) 
    model = AUDeepfakeDetector()
    
    preds = model(mock_au_input)
    print("Predictions shape:", preds.shape) # 기대값: (8, 1)

Predictions shape: torch.Size([8, 1])


In [ ]:
# 모델 훈련
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AUDeepfakeDetector().to(device)

criterion = nn.BCELoss()  # 이진 분류
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 150

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for au_batch, labels in dataloader:
        au_batch = au_batch.to(device)
        labels = labels.float().to(device).unsqueeze(1)  # (B,) -> (B, 1)
        
        optimizer.zero_grad()
        outputs = model(au_batch)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / len(dataloader):.4f}")

print("Training completed.")

Epoch 1/10, Loss: 0.6252
Epoch 2/10, Loss: 0.1283
Epoch 3/10, Loss: 0.0198
Epoch 4/10, Loss: 0.0170
Epoch 5/10, Loss: 0.0185
Epoch 6/10, Loss: 0.0055
Epoch 7/10, Loss: 0.0143
Epoch 8/10, Loss: 0.0146
Epoch 9/10, Loss: 0.0125
Epoch 10/10, Loss: 0.0101
Training completed.


In [ ]:
# 하이퍼파라미터 튜닝 (Grid Search)
from sklearn.metrics import f1_score
import itertools

# 튜닝할 파라미터
learning_rates = [0.001, 0.01]
hidden_dims = [32, 64, 128]

best_f1 = 0
best_params = {}

for lr, hd in itertools.product(learning_rates, hidden_dims):
    print(f"Testing lr={lr}, hidden_dim={hd}")
    model = AUDeepfakeDetector(hidden_dim=hd).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # 간단한 훈련 (10 에폭으로 튜닝용)
    for epoch in range(10):
        model.train()
        for au_batch, labels in dataloader:
            au_batch = au_batch.to(device)
            labels = labels.float().to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(au_batch)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # 평가
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for au_batch, labels in dataloader:
            au_batch = au_batch.to(device)
            outputs = model(au_batch)
            preds = (outputs > 0.5).float().cpu().numpy().flatten()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    f1 = f1_score(all_labels, all_preds)
    print(f"F1 Score: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_params = {'lr': lr, 'hidden_dim': hd}

print(f"Best F1: {best_f1:.4f}, Best Params: {best_params}")
# 최고 파라미터로 모델 재훈련 및 평가 (아래 셀에서 사용)

In [7]:
# 모델 평가 및 실시간성 테스트
import time
from sklearn.metrics import accuracy_score, f1_score

model.eval()
all_preds = []
all_labels = []

# 추론 시간 측정
start_time = time.time()
with torch.no_grad():
    for au_batch, labels in dataloader:
        au_batch = au_batch.to(device)
        outputs = model(au_batch)
        preds = (outputs > 0.5).float()
        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(labels.numpy())

inference_time = time.time() - start_time
avg_inference_time = inference_time / len(dataset)

print(f"Average inference time per sample: {avg_inference_time:.4f} seconds")
print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1 Score: {f1_score(all_labels, all_preds):.4f}")

# 실시간성: 30 FPS 기준으로 0.033초 이하 필요
if avg_inference_time < 0.033:
    print("실시간성 보장됨.")
else:
    print("실시간성 개선 필요.")

Average inference time per sample: 0.0004 seconds
Accuracy: 1.0000
F1 Score: 0.0000
실시간성 보장됨.


/Users/soyun/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
# 가벼운 모델 테스트
light_model = LightweightAUDetector().to(device)
light_optimizer = optim.Adam(light_model.parameters(), lr=0.001)

# 빠른 훈련
for epoch in range(5):
    light_model.train()
    for au_batch, labels in dataloader:
        au_batch = au_batch.to(device)
        labels = labels.float().to(device).unsqueeze(1)
        
        light_optimizer.zero_grad()
        outputs = light_model(au_batch)
        loss = criterion(outputs, labels)
        loss.backward()
        light_optimizer.step()

# 평가
light_model.eval()
start_time = time.time()
with torch.no_grad():
    for au_batch, labels in dataloader:
        au_batch = au_batch.to(device)
        outputs = light_model(au_batch)
light_inference_time = time.time() - start_time
light_avg_time = light_inference_time / len(dataset)

print(f"Lightweight model average inference time: {light_avg_time:.4f} seconds")
if light_avg_time < 0.033:
    print("가벼운 모델 실시간성 보장됨.")
else:
    print("가벼운 모델도 개선 필요.")

Lightweight model average inference time: 0.0001 seconds
가벼운 모델 실시간성 보장됨.


In [10]:
# 모델 최적화: TorchScript 변환
light_model.eval()
example_input = torch.rand(1, 64, 17).to(device)
scripted_model = torch.jit.trace(light_model, example_input)
scripted_model.save("lightweight_au_detector.pt")

print("모델이 TorchScript로 변환되어 저장됨. 배포 시 사용 가능.")

모델이 TorchScript로 변환되어 저장됨. 배포 시 사용 가능.
